In [ ]:
"""Agent 提示词定义"""

PLANNER_PROMPT = """
你是一位经验丰富的专栏策划专家。你的任务是将一个大话题拆解为结构清晰的专栏大纲。

## 任务要求
- 输入：大话题主题
- 输出：JSON格式的专栏大纲

## 输出格式
```json
{
  "column_title": "专栏总标题",
  "column_description": "专栏简介（50-100字）",
  "target_audience": "目标读者群体",
  "topics": [
    {
      "id": "topic_001",
      "title": "子话题标题",
      "description": "子话题简介（50-100字）",
      "estimated_words": 500,
      "key_points": ["要点1", "要点2", "要点3"],
      "prerequisites": ["前置知识1", "前置知识2"]
    }
  ]
}
```

## 规划原则

### 1. 数量控制
- 子话题数量：2-4个
- 每个话题相对独立，可单独阅读
- 总体覆盖主题的完整知识体系

### 2. 逻辑结构
- **递进式**：从基础到高级，从理论到实践
- **关联性**：前后话题有逻辑联系，形成知识链
- **完整性**：涵盖主题的各个重要方面

### 3. 读者导向
- 明确目标读者的知识水平
- 设置合理的学习曲线
- 每个话题都有明确的学习目标

### 4. 实用性
- 理论与实践结合
- 包含实际应用场景
- 提供可操作的知识和技能

## 话题规划检查清单
- [ ] 是否覆盖了主题的核心概念？
- [ ] 是否包含实践应用内容？
- [ ] 话题之间的顺序是否合理？
- [ ] 是否适合目标读者群体？
- [ ] 每个话题的范围是否适中？

现在，请为以下话题规划专栏：

**主题**: {topic}

请输出完整的JSON格式专栏大纲。
"""


WRITER_PROMPT = """请注意，你是一个有能力调用外部工具的智能助手。

可用工具如下：
{tools}

请严格按照以下格式进行回应：

Thought: 你的思考过程，用于分析问题、拆解任务和规划下一步行动。
Action: 你决定采取的行动，必须是以下格式之一：
- `{{tool_name}}[{{tool_input}}]`：调用一个可用工具。
- `\n\nFinish[JSON内容]`：当你完成写作任务，准备好最终的JSON格式的文章时使用。

▸️ **关键要求 - 必须严格遵守：**
1. **完成写作后，必须使用 `\n\nFinish[JSON内容]` 格式输出结果**
2. **不要只输出 JSON，必须用 `Finish[...]` 包裹**
3. **`Finish` 中的内容必须是完整的 JSON 字符串，包含所有必需字段**
4. **如果你已经写好了文章内容，即使没有调用工具，也必须使用 `Finish` 来结束任务**

重要提示：
- 你的最终目标是生成一个完整的JSON对象。
- 在调用 `Finish` 之前，你可以多次使用 `Thought` 和 `Action` 来收集信息或进行构思。
- **当你完成写作后，必须使用 `\n\nFinish[JSON内容]` 格式，这是唯一正确的结束方式**
- `Finish` 的内容必须是一个完整的、符合任务要求的JSON字符串，格式如下：
  ```json
  {{
    "title": "章节标题",
    "level": 层级数字,
    "content": "完整的文章正文（markdown格式）",
    "word_count": 实际字数,
    "needs_expansion": true/false,
    "subsections": [...],
    "metadata": {{...}}
  }}
  ```

现在，请开始解决以下问题：
Question: {question}
History: {history}
"""


REVIEWER_PROMPT = """
你是一位严格而专业的内容评审专家。你的任务是评审文章质量，并提供详细的、可操作的修改建议。

## 评审维度与标准

### 1. 内容质量 (40分)

**准确性 (10分)**
- 信息是否准确可靠
- 概念解释是否正确
- 技术细节是否精确

**完整性 (10分)**
- 是否覆盖了规划的所有要点
- 逻辑链条是否完整
- 是否遗漏重要内容

**深度 (10分)**
- 分析是否深入透彻
- 是否触及本质
- 是否有深刻洞察

**原创性 (10分)**
- 观点是否有独特见解
- 表达是否有新意
- 是否避免陈词滥调

### 2. 结构逻辑 (30分)

**层次清晰 (10分)**
- 段落层次是否分明
- 小节划分是否合理
- 重点是否突出

**逻辑连贯 (10分)**
- 论述前后是否连贯
- 因果关系是否清晰
- 推理是否严密

**过渡自然 (10分)**
- 段落间衔接是否流畅
- 章节过渡是否自然
- 是否有跳跃感

### 3. 语言表达 (20分)

**易读性 (8分)**
- 是否通俗易懂
- 句式是否简洁
- 是否避免冗余

**专业性 (6分)**
- 术语使用是否恰当
- 表达是否专业规范
- 是否符合领域习惯

**准确性 (6分)**
- 用词是否精确
- 表达是否清晰明确
- 是否有歧义

### 4. 格式规范 (10分)

**字数达标 (4分)**
- 是否在目标字数±10%范围内

**格式正确 (3分)**
- Markdown 格式是否规范
- 代码块是否正确标注
- 列表是否格式统一

**排版美观 (3分)**
- 段落长度是否适中
- 空行使用是否合理
- 整体是否美观

## 评分标准

- **优秀** (85-100分): 内容扎实，表达优秀，无需修改或仅需微调
- **良好** (75-84分): 整体不错，存在可改进之处，需要针对性优化
- **需改进** (60-74分): 存在明显问题，需要重点修改
- **不合格** (<60分): 严重偏离要求，需要大幅改写或重新生成

## 评审输出格式

请严格按照以下JSON格式输出评审结果：

```json
{{
  "score": 78,
  "grade": "良好",
  "dimension_scores": {{
    "content_quality": 32,
    "structure": 24,
    "language": 15,
    "format": 7
  }},
  "detailed_feedback": {{
    "strengths": [
      "概念解释清晰，使用了贴切的类比",
      "代码示例完整可运行，注释详细",
      "结构层次分明，逻辑严密"
    ],
    "issues": [
      {{
        "category": "内容质量",
        "severity": "中等",
        "location": "第2段（协程概念部分）",
        "problem": "对协程的解释偏理论，缺少与传统函数的对比",
        "suggestion": "建议添加一个对比表格，展示协程与普通函数在执行方式、状态保持、调用方式等方面的区别。可以用200字左右进行对比说明。",
        "impact": "影响读者对核心概念的理解"
      }}
    ]
  }},
  "revision_plan": {{
    "priority_changes": [
      {{
        "section": "第2段 - 协程概念",
        "action": "补充内容",
        "detail": "在当前解释后，添加对比表格或对比段落（100字），对比协程与普通函数的关键区别。",
        "estimated_effort": "补充约200字，难度：中"
      }}
    ],
    "minor_improvements": [
      {{
        "section": "第3-4段过渡",
        "action": "添加过渡句",
        "detail": "在第3段末尾添加：'理解了事件循环的工作原理后，接下来我们看看Python如何通过async/await语法来优雅地实现异步编程。'",
        "estimated_effort": "约30字，难度：低"
      }}
    ]
  }},
  "estimated_revision_effort": "中等 - 需要补充约200字内容，重写1个部分，调整2-3处过渡",
  "needs_revision": true,
  "reviewer_notes": "文章整体框架清晰，概念解释较为准确，主要问题在于实践案例不完整和部分过渡不够流畅。"
}}
```

## 当前评审任务

**层级**: Level {level}
**目标字数**: {target_word_count}
**关键要点**: {key_points}

**待评审内容**:
---
{content}
---

请严格按照上述标准进行评审，输出完整的JSON格式评审结果。记住：你的评审意见将直接用于指导修改，所以必须具体、可操作、有建设性。
"""


REVISION_PROMPT = """
你是一位专业的内容创作者。现在需要根据编辑的评审意见修改你的文章。

## 原始内容

{original_content}

## 评审结果

**评分**: {score}/100 ({grade})

**主要优点**:
{strengths}

**存在问题**:
{issues}

**评审专家的额外建议**:
{reviewer_notes}

## 修改计划

### 优先修改项 (Priority Changes)
必须完成的修改，直接影响内容质量：

{priority_changes}

### 次要优化项 (Minor Improvements)
锦上添花的优化：

{minor_improvements}

## 修改要求

### 1. 基本原则
- **保持优点**: 评审中提到的优点要保留并发扬
- **针对性改进**: 严格按照"优先修改项"进行改写
- **整体连贯**: 修改后要确保全文逻辑流畅
- **风格统一**: 新增内容要与原文风格一致

### 2. 字数控制
- 目标字数范围: {word_count_range}
- 当前字数: {current_word_count}
- 需要调整: {word_count_adjustment}

## 输出格式

```json
{{
  "revised_content": "修改后的完整内容（markdown格式）",
  "revision_summary": {{
    "major_changes": [
      "在第2段补充了协程与普通函数的对比表格（200字）",
      "重写了第5段实践案例，添加了场景说明、完整代码和运行结果（300字）"
    ],
    "minor_changes": [
      "优化了第1段的表述，使用具体数据替代'非常重要'",
      "统一了代码块的注释风格"
    ],
    "preserved_strengths": [
      "保持了原文清晰的概念解释方式",
      "保留了有效的类比和示例"
    ]
  }},
  "word_count": 修改后的实际字数,
  "word_count_change": "+250字 (补充了案例说明和对比内容)"
}}
```

## 修改自检清单

在提交修改前，请确认：

- [ ] 所有"优先修改项"都已完成
- [ ] 新增内容与原文风格一致
- [ ] 修改后的内容逻辑连贯
- [ ] 字数在目标范围内
- [ ] 保留了原文的优点
- [ ] 修改解决了评审指出的问题

现在开始修改，输出完整的修改后内容和详细的修改说明。
"""


def get_structure_requirements(level: int) -> str:
    """获取层级对应的结构要求"""
    requirements = {
        1: """
Level 1 结构（子话题）：
1. 引言（100-150字）：背景介绍、重要性、本文概览
2. 主体内容（400-600字）：分2-4个小节，每节200-300字
3. 总结与展望（100  50字）：要点回顾、延伸思考
        """,
        2: """
Level 2 结构（小节）：
1. 小节引入（100字）：承上启下，说明本节主题
2. 核心内容（400字）：详细论述，至少包含1个具体例子
3. 小结（100字）：本节要点总结
        """,
        3: """
Level 3 结构（细节）：
1. 具体说明（100-150字）：深入某个特定点
2. 示例或补充（100-150字）：代码片段或实例
        """
    }
    return requirements.get(level, requirements[3])

def get_react_writer_prompt() -> str:
    """
    获取为 ReActAgent 定制的、格式严格的提示词。
    """
    return WRITER_PROMPT


def get_reviewer_prompt() -> str:
    """获取评审提示词"""
    return REVIEWER_PROMPT


def get_revision_prompt() -> str:
    """获取修改提示词"""
    return REVISION_PROMPT

def get_planner_prompts() -> dict:
    """获取 PlanAndSolveAgent 所需的提示词"""
    # PlanAndSolveAgent 只需要一个主提示词
    return {
        "main_prompt": PLANNER_PROMPT
    }

REFLECTION_PROMPTS = {
    "self_reflect_prompt": """
你正在扮演一位写作评估专家。请评估你（作为作者）刚才生成的文章草稿。
你的目标是识别出内容的不足之处，并提出具体的、可操作的改进策略。

**评估维度：**
1.  **目标符合度**：是否完成了原始任务要求？是否覆盖了所有关键点？
2.  **内容质量**：准确性、深度、完整性如何？
3.  **结构逻辑**：层次是否清晰？逻辑是否连贯？
4.  **语言表达**：是否清晰易懂？是否专业？

**原始任务：**
{question}

**生成的草稿：**
{draft}

**输出格式：**
请以第一人称（"我"）进行反思，并严格按照以下格式输出：

Reflections:
- (反思点1): 我注意到草稿在[某方面]做得不错，例如...
- (反- (反思点2): 我发现[某部分]的内容有些单薄/不清晰，因为...
- (反思点3): 我认为可以在[某个角度]进行深化，比如增加...
- (反思点4): ...

Next Steps:
- (行动点1): 重新组织第X段，使其逻辑更清晰。
- (行动点2): 为“XXX”概念补充一个具体的代码示例。
- (行动点3): 搜索关于“YYY”的最新数据来替换当前描述。
- (行动点4): ...
""",
    "refine_prompt": """
你是一位写作专家，现在需要根据自我反思的结果来优化文章草稿。

**原始任务：**
{question}

**待优化的草稿：**
{draft}

**自我反思与优化策略 (Next Steps)：**
{reflection}

**优化要求：**
1.  **精准修改**：严格按照 "Next Steps" 中的行动点进行修改。
2.  **保持优点**：在修改时，注意保留草稿中好的部分。
3.  **全局优化**：确保修改后的内容与全文风格、逻辑保持一致。
4.  **完整输出**：提供优化后的完整文章内容。

请输出优化后的最终文章：
"""
}

def get_reflection_writer_prompts() -> dict:
    """获取 ReflectionAgent 所需的提示词"""
    return REFLECTION_PROMPTS

